# NeurIPS Revision GPU Session 1

Zero-dollar Kaggle notebook for the first hosted-GPU batch.

Runs:

1. Fix 1 generation + analysis for the causal coherence intervention.
2. Fix 5 coherence-preserving noise slope.
3. Fix 11 RAPTOR full table.

Expected wall-clock on a free Kaggle/Colab GPU: about 3.5-8.5 hours plus setup. Enable Internet and GPU before running.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Saket-Maganti/rag-hallucination-detection.git"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/rag-hallucination-detection")
OUTPUT_ZIP = Path("/kaggle/working/revision_session1_outputs.zip")

print("repo:", REPO_URL)
print("branch:", BRANCH)
print("output:", OUTPUT_ZIP)

In [ ]:
!nvidia-smi || true
!python --version
!df -h /kaggle/working

In [ ]:
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
print("cwd:", Path.cwd())

## Install Ollama and Pull Mistral

This is the only large setup step. It may take 5-20 minutes depending on Kaggle network speed.

In [ ]:
%%bash
set -euo pipefail

if ! command -v zstd >/dev/null 2>&1; then
  apt-get update -y
  apt-get install -y zstd
fi

if ! command -v ollama >/dev/null 2>&1; then
  curl -fsSL https://ollama.com/install.sh | sh
fi

if ! pgrep -x ollama >/dev/null 2>&1; then
  nohup ollama serve > /kaggle/working/ollama.log 2>&1 &
  sleep 8
fi

ollama pull mistral
ollama list

## Python Dependencies

Keep this cell separate so a dependency failure does not waste model-pull time.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/rag-hallucination-detection
python -m pip install -U pip
if [ -f requirements.txt ]; then
  python -m pip install -r requirements.txt
else
  echo "No root requirements.txt found; installing explicit revision dependencies"
fi
python -m pip install -e pip-package
python -m pip install scipy pandas numpy scikit-learn matplotlib seaborn tabulate
python -m pip install sentence-transformers transformers torch datasets langchain-ollama

## Preflight

This checks that the revision scripts import before starting multi-hour runs.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/rag-hallucination-detection
mkdir -p logs/revision
python -m py_compile \
  experiments/fix_01_causal_matched_pairs.py \
  experiments/fix_05_coherence_preserving_noise.py \
  experiments/fix_11_raptor_full_table.py \
  experiments/revision_utils.py \
  src/ragas_scorer.py src/vectara_hem_scorer.py
python experiments/fix_01_causal_matched_pairs.py --help >/tmp/fix01_help.txt
python experiments/fix_05_coherence_preserving_noise.py --help >/tmp/fix05_help.txt
python experiments/fix_11_raptor_full_table.py --help >/tmp/fix11_help.txt
echo "preflight OK"

## Fix 1: Causal Intervention Generation + Analysis

The primary matched-pair construction should already be present in `data/revision/fix_01/matched_pairs.csv`. If it is missing, this notebook reconstructs it before generation.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/rag-hallucination-detection
mkdir -p logs/revision

if [ ! -s data/revision/fix_01/matched_pairs.csv ]; then
  echo "matched_pairs.csv missing; reconstructing Fix 1 pairs"
  python experiments/fix_01_causal_matched_pairs.py \
    --stage construct \
    --dataset squad \
    --n_target 200 \
    --seed 42 \
    --max_contexts 400 \
    --candidate_limit 400 \
    --run_tag primary_n200 \
    2>&1 | tee logs/revision/fix_01_construct_session1.log
else
  echo "using existing Fix 1 matched pairs"
fi

python experiments/fix_01_causal_matched_pairs.py \
  --stage generate \
  --backend ollama \
  --model mistral \
  --resume \
  --save_every 10 \
  --progress_every 10 \
  2>&1 | tee logs/revision/fix_01_generate_session1.log

python experiments/fix_01_causal_matched_pairs.py \
  --stage analyze \
  2>&1 | tee logs/revision/fix_01_analyze_session1.log

## Fix 5: Coherence-Preserving Noise Slope

Run this after Fix 1. Estimated free-GPU runtime: 1.5-4 hours.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/rag-hallucination-detection
mkdir -p logs/revision
python experiments/fix_05_coherence_preserving_noise.py \
  --n 200 \
  --seed 42 \
  --backend ollama \
  --model mistral \
  --max_contexts 300 \
  --n_noise 1 2 3 \
  --save_every 25 \
  2>&1 | tee logs/revision/fix_05_noise_slope_session1.log

## Fix 11: RAPTOR Full Table

Estimated free-GPU runtime: 1-2.5 hours.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/rag-hallucination-detection
mkdir -p logs/revision
python experiments/fix_11_raptor_full_table.py \
  --datasets squad pubmedqa hotpotqa \
  --n 100 \
  --backend ollama \
  --model mistral \
  --max_contexts 150 \
  --raptor_clusters 6 \
  2>&1 | tee logs/revision/fix_11_raptor_full_table_session1.log

## Package Outputs

Download `/kaggle/working/revision_session1_outputs.zip` from the Kaggle file browser and copy it back into the local repo.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/rag-hallucination-detection
rm -f /kaggle/working/revision_session1_outputs.zip
zip -r /kaggle/working/revision_session1_outputs.zip \
  data/revision/fix_01 results/revision/fix_01 \
  data/revision/fix_05 results/revision/fix_05 \
  data/revision/fix_11 results/revision/fix_11 \
  logs/revision \
  REVISION_SUMMARY.md REVISION_RUNBOOK.md
ls -lh /kaggle/working/revision_session1_outputs.zip